In [1]:
"""k-NN (emlearn) - varredura, geracao da lib C e verificacao exata.

A lib entregue (lib/knob-senses-knn) e' escrita pela celula de geracao: re-executar o
notebook reproduz os arquivos byte a byte (mesma semente). O conferidor g++ prova que o
C do firmware reproduz o emulador bit a bit (em mW)."""

import subprocess
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'models' / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import emlearn

from common import contract, data, paths, verify

import knn

RAIZ = paths.ROOT

SEMENTE = contract.SEED
FEATURES = contract.RF_FEATURES
ALVO = contract.TARGET

np.random.seed(SEMENTE)

print("emlearn :", emlearn.__version__)
print("features:", ", ".join(FEATURES))
print("g++ host:", subprocess.run([verify.GXX, "--version"], capture_output=True,
                                 text=True).stdout.splitlines()[0])
print("entrega :", paths.LIB_KNN.relative_to(RAIZ))

emlearn : 0.23.2
features: temperatura_ambiente, ghit, jacques
g++ host: g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
entrega : lib/knob-senses-knn


In [2]:
treino = data.load_split("train")
calibracao = data.load_split("calibration")
teste = data.load_split("test")

# Sanidade: o csv precisa estar em GRANDEZA FISICA. Se alguem voltar a normalizar no
# prep, estes limites quebram AQUI - e nao em campo, com o firmware normalizando duas
# vezes e a previsao colapsando num valor fixo.
for nome, d in (("train", treino), ("calibration", calibracao), ("test", teste)):
    data.check_physical_scale(nome, d)

print(f"treino {len(treino)} | calibracao {len(calibracao)} | teste {len(teste)}")

treino 14090 | calibracao 2023 | teste 4039


In [3]:
# Media e desvio SO do split de treino (o mesmo contrato do RF).
MEDIA, DESVIO = data.fit_normalization(treino)

X_tre = treino[list(FEATURES)]
y_tre = treino[ALVO].to_numpy()
X_tes = teste[list(FEATURES)]
y_tes = teste[ALVO].to_numpy()

# Quantizacao int16 (imposta pelo emlearn): z -> lrintf(z*1000) clampado em +-8. O
# clamp nao e' estetica: garante soma de quadrados < 2**30, onde o eml_isqrt do
# emlearn travaria (o `q <<= 2` estoura o int32). Os rotulos viram miliwatts.
Zq_tre = knn.quantiza_z(X_tre, MEDIA, DESVIO)
yq_tre = knn.quantiza_alvo(y_tre)
Zq_tes = knn.quantiza_z(X_tes, MEDIA, DESVIO)

print("quantizado: treino", Zq_tre.shape, "| z min/max", Zq_tre.min(), Zq_tre.max())

quantizado: treino (14090, 3) | z min/max -2835 2715


In [4]:
# Tamanho x acuracia: quantos prototipos e quantos vizinhos. sklearn sobre os dados
# QUANTIZADOS (rapido); a exatidao fica com o emulador mais abaixo. RAM = scratch do
# emlearn (8 B por prototipo) - o custo de usar o pipeline dele como vem.
t0 = time.perf_counter()
tabela = knn.sweep(Zq_tre, yq_tre, Zq_tes, y_tes,
                   n_opcoes=(1000, 2000, 4000, len(treino)), k_opcoes=knn.SWEEP_K)
print(f"{time.perf_counter() - t0:.1f} s\n")
print(tabela.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

0.3 s

 prototipos  k    mae     r2  flash_kb   ram_kb
       1000  1 1.3783 0.9093    7.8125   7.8125
       1000  3 1.2808 0.9232    7.8125   7.8125
       1000  5 1.2440 0.9284    7.8125   7.8125
       1000  9 1.2354 0.9303    7.8125   7.8125
       1000 15 1.2150 0.9324    7.8125   7.8125
       2000  1 1.4212 0.9040   15.6250  15.6250
       2000  3 1.3371 0.9182   15.6250  15.6250
       2000  5 1.3160 0.9208   15.6250  15.6250
       2000  9 1.2892 0.9245   15.6250  15.6250
       2000 15 1.2684 0.9275   15.6250  15.6250
       4000  1 1.4071 0.9038   31.2500  31.2500
       4000  3 1.3327 0.9180   31.2500  31.2500
       4000  5 1.3169 0.9211   31.2500  31.2500
       4000  9 1.3081 0.9229   31.2500  31.2500
       4000 15 1.2915 0.9251   31.2500  31.2500
      14090  1 1.4022 0.9051  110.0781 110.0781
      14090  3 1.3430 0.9170  110.0781 110.0781
      14090  5 1.3199 0.9204  110.0781 110.0781
      14090  9 1.3120 0.9222  110.0781 110.0781
      14090 15 1.3075 0.9234  110

In [5]:
# A regra do spec: (prototipos, k) crescente; vence a primeira config a <= 5 % do
# melhor MAE - o menor conjunto que nao custa acuracia de verdade.
K_ESCOLHIDO, N_ESCOLHIDO, PORQUE = knn.escolhe_config(tabela)
print(f"escolha: k = {K_ESCOLHIDO}, prototipos = {N_ESCOLHIDO}")
print(PORQUE)

escolha: k = 5, prototipos = 1000
MAE 1.2440 kW (melhor da varredura: 1.2150; limite de 5 %: 1.2757) com o menor conjunto/k que atende


In [6]:
linha = tabela[(tabela["k"] == K_ESCOLHIDO) & (tabela["prototipos"] == N_ESCOLHIDO)].iloc[0]
escritos = knn.generate_c_lib(X_tre, y_tre, MEDIA, DESVIO,
                              mae=linha["mae"], r2=linha["r2"],
                              k=K_ESCOLHIDO, n=N_ESCOLHIDO)
for caminho in sorted(escritos):
    print(f"  {caminho.relative_to(RAIZ)!s:52} {escritos[caminho]:,} bytes")
print("\nOK: nenhum marcador sobrou nos arquivos gerados")

  lib/knob-senses-knn/README.md                        1,613 bytes
  lib/knob-senses-knn/library.json                     505 bytes
  lib/knob-senses-knn/normalizacao.json                968 bytes
  lib/knob-senses-knn/src/knob_senses_knn.cpp          3,229 bytes
  lib/knob-senses-knn/src/knob_senses_knn.h            2,208 bytes

OK: nenhum marcador sobrou nos arquivos gerados


In [7]:
conferidor = verify.write_checker(dest=paths.SCRATCH / "knn_checker.cpp",
                                     header="knob_senses_knn.h", func="knn_pca_prever",
                                     n_macro="KNOB_KNN_N_FEATURES",
                                     n_features=len(FEATURES))
exe = verify.compile_host(conferidor, paths.SCRATCH / "knn_checker.exe",
                             lib_src=paths.LIB_KNN_SRC, impl="knob_senses_knn.cpp")
print("g++: compilou o conferidor junto com o knob_senses_knn.cpp da lib")

# As linhas de teste em BRUTO pelo C...
pred_c = np.asarray(verify.predict(exe, X_tes.to_numpy()), dtype=np.float64)
pred_c_mw = np.rint(pred_c * 1000).astype(np.int64)

# ...e o emulador replicando o MESMO algoritmo (float32 da consulta incluso):
idx = knn._prototipos(len(treino), N_ESCOLHIDO)
emu = knn.preve_emulado_lote(knn.quantiza_z(X_tre.iloc[idx], MEDIA, DESVIO),
                             knn.quantiza_alvo(y_tre[idx]),
                             knn.quantiza_z_c(X_tes, MEDIA, DESVIO), K_ESCOLHIDO)
assert np.array_equal(pred_c_mw, emu), "o C divergiu do emulador - bug de verdade"
print(f"OK: {len(pred_c)} linhas identicas ao emulador (bit a bit, em mW)")

# Referencia arbitraria: sklearn nos mesmos ints. As diferencas sao EMPATES do
# floor-sqrt do emlearn (a distancia inteira empata mais que o float do sklearn).
from sklearn.neighbors import KNeighborsRegressor
sk = KNeighborsRegressor(n_neighbors=K_ESCOLHIDO, algorithm="brute", metric="euclidean",
                         weights="uniform").fit(
    knn.quantiza_z(X_tre.iloc[idx], MEDIA, DESVIO).astype(np.float64),
    knn.quantiza_alvo(y_tre[idx]).astype(np.float64))
dif = np.abs(pred_c - sk.predict(knn.quantiza_z(X_tes, MEDIA, DESVIO).astype(np.float64)) / 1000)
print(f"vs sklearn: {int((dif * 1000 > 0.75).sum())} de {len(dif)} linhas > 0,75 mW "
      f"(maior {dif.max():.4f} kW) - empates de fronteira, nao erro")

g++: compilou o conferidor junto com o knob_senses_knn.cpp da lib
OK: 4039 linhas identicas ao emulador (bit a bit, em mW)
vs sklearn: 59 de 4039 linhas > 0,75 mW (maior 1.4780 kW) - empates de fronteira, nao erro


In [8]:
erro = np.abs(pred_c - y_tes)
amplitude = float(pred_c.max() - pred_c.min())
print(f"o caminho inteiro, nas {len(X_tes)} linhas de test.csv:")
print(f"  MAE        : {erro.mean():.4f} kW")
print(f"  amplitude  : {amplitude:.4f} kW")
print(f"  primeira   : real {y_tes[0]:.4f}  previsto {pred_c[0]:.4f}")

# Duas guardas: contrato quebrado (normalizacao/quantizacao dupla) e previsao constante.
assert erro.mean() < 0.35, f"MAE {erro.mean():.4f} kW pelo caminho da rede - o contrato quebrou"
assert amplitude > 0.1, "amplitude baixa: previsao praticamente constante"
print("\nOK: o caminho csv -> quantiza -> lib devolve previsao util")

o caminho inteiro, nas 4039 linhas de test.csv:
  MAE        : 1.2444 kW
  amplitude  : 21.7220 kW
  primeira   : real 3.5500  previsto 2.6800


AssertionError: MAE 1.2444 kW pelo caminho da rede - o contrato quebrou

In [ ]:
medida = verify.measure_xtensa(lib_src=paths.LIB_KNN_SRC,
                                  impl="knob_senses_knn.cpp",
                                  obj_file=paths.SCRATCH / "knob_senses_knn.o")
flash, ram = medida["flash"], medida["ram"]
print(f"  FLASH  {flash:7} bytes  ({flash / 1024:.1f} KB)")
print(f"  RAM    {ram:7} bytes  ({ram / 1024:.1f} KB)  = 8 B x {N_ESCOLHIDO} prototipos + bss")
print(f"  para comparar: o RF ocupa ~68 KB de flash e 0 de RAM")

In [ ]:
print("k-NN (emlearn) - procedencia")
print(f"  k = {K_ESCOLHIDO} | prototipos = {N_ESCOLHIDO} | semente = {SEMENTE}")
print(f"  MAE {erro.mean():.4f} kW | R2 {linha['r2']:.4f} no split de teste")
print(f"  flash ~{flash / 1024:.1f} KB | RAM ~{ram / 1024:.1f} KB")
print("  entrega:", paths.LIB_KNN.relative_to(RAIZ))